# PKG Notebook — Aggregated Network (Jan–Nov 2025) & Subrogation Candidate Extraction

Loads monthly snapshot CSVs, builds an aggregated payment graph over the
chosen window, parses NAICS code/description + multi-digit rollups, and
sets up tooling to identify and inspect candidate insurance-subrogation
sub-networks (NAICS + name filtering, first-order ego graph, payer/payee
and reciprocal-flow analysis).

In [ ]:
import time
from pathlib import Path
from typing import Dict, List, Optional, Set

import numpy as np
import pandas as pd

## 0. Config

In [ ]:
DATA_DIR = Path("../data")
FILE_PREFIX = "cust_"
START_YM = "2025-01"
END_YM = "2025-11"

RAW_DTYPES = {
    "source": "string",
    "source_name": "string",
    "source_naics": "string",
    "amount": "float64",
    "volume": "float64",   # transaction count per edge per month; float tolerates NaN
    "dest": "string",
    "dest_name": "string",
    "dest_naics": "string",
}

def month_range(start_ym: str, end_ym: str) -> List[str]:
    start = pd.Timestamp(start_ym + "-01")
    end = pd.Timestamp(end_ym + "-01")
    return pd.date_range(start, end, freq="MS").strftime("%Y-%m").tolist()

MONTHS = month_range(START_YM, END_YM)
print(f"Months to load ({len(MONTHS)}): {MONTHS[0]} .. {MONTHS[-1]}")

# --- Entity-type config: insurers and law firms -------------------------
# Insurers: NAICS 524x (carriers, TPAs, adjusters, brokers). Law firms:
# NAICS 5411x (Legal Services -- 541110 lawyers, 541191 title/settlement
# offices, 541199 other legal). Because NAICS is frequently missing or a
# placeholder here, each class also has a name-pattern fallback, and law
# firms carry a separate "trust account" flag -- the strongest
# subrogation-specific signal, since WC/GL recoveries move through
# attorney/plaintiff-firm trust (IOLTA) accounts and still fire when NAICS
# is absent. Patterns are deliberately high-precision; widen/narrow them
# against your real name distribution.
INSURER_NAICS3 = ["524"]
LAW_FIRM_NAICS4 = ["5411"]

INSURER_NAME_PATTERN = r"\b(?:insurance|assurance|casualty|indemnity|reinsurance|underwriters?)\b"

# STRONG: tokens that denote legal PRACTICE and little else.
# Deliberately excludes bare "LLP" / "PC" / "PLLC" -- those are generic
# entity-form suffixes shared with accounting and medical practices
# ("PEARSON HARDMAN ACCOUNTING LLP", "RIVERBEND FAMILY MEDICINE PC") and
# generate false positives on their own.
LAW_FIRM_NAME_PATTERN = (
    r"\b(?:law\s+offices?|law\s+firm|law\s+group|law\s+center|attorneys?|"
    r"legal\s+services|legal\s+group|counsell?ors?\s+at\s+law|"
    r"esq(?:uire)?|barristers?|solicitors?)\b"
)
TRUST_ACCOUNT_PATTERN = (
    r"\b(?:iolta|client\s+trust|attorney\s+trust|lawyers?\s+trust|"
    r"trust\s+account|client\s+funds)\b"
)

# Persistence: how many DISTINCT months a pair must recur to count as a
# "persistent" relationship. Default ~half the window; tune per window.
N_WINDOW_MONTHS = len(MONTHS)
PERSISTENCE_MIN_MONTHS = max(3, N_WINDOW_MONTHS // 2)
print(f"Persistence threshold: {PERSISTENCE_MIN_MONTHS} of {N_WINDOW_MONTHS} months")

# Directional thresholds for insurer<->law-firm pairs (section 12b). A pair
# whose money runs >=80% firm->insurer reads as recovery inflow; <=20% reads
# as outbound defense/funding; in between is two-way recovery counsel.
RECOVERY_RATIO_HIGH = 0.80
RECOVERY_RATIO_LOW = 0.20

# Visualisation hub suppression. A float in (0,1] is a QUANTILE of the
# subgraph's degree distribution (0.95 = drop the top 5% most-connected
# nodes); an int is an absolute degree cap. Quantiles are used because an
# absolute threshold doesn't transfer between a 10-node ego view and a
# 10M-node graph. Suppression is skipped entirely on subgraphs smaller than
# HUB_SUPPRESS_MIN_NODES, where a skewed-degree notion of "hub" is
# meaningless and would just empty the picture. Set None to disable.
HUB_DEGREE_CAP = 0.95
HUB_SUPPRESS_MIN_NODES = 30

## 1. Locate files & load, tagging each row with its source month

In [ ]:
def find_missing_files(months, data_dir=DATA_DIR, prefix=FILE_PREFIX):
    missing = []
    for ym in months:
        p = data_dir / f"{prefix}{ym}.csv"
        if not p.exists():
            missing.append(str(p))
    return missing

missing = find_missing_files(MONTHS)
if missing:
    print(f"WARNING: {len(missing)} file(s) not found, will be skipped:")
    for m in missing:
        print(f"  - {m}")

In [ ]:
def load_month(ym: str, data_dir=DATA_DIR, prefix=FILE_PREFIX) -> pd.DataFrame:
    path = data_dir / f"{prefix}{ym}.csv"
    df = pd.read_csv(path, dtype=RAW_DTYPES)
    df["year_month"] = ym
    return df

frames = []
t0 = time.time()
for ym in MONTHS:
    path = DATA_DIR / f"{FILE_PREFIX}{ym}.csv"
    if not path.exists():
        continue
    t1 = time.time()
    df = load_month(ym)
    frames.append(df)
    print(f"  {ym}: {len(df):>10,} rows  ({time.time() - t1:5.1f}s)")

if not frames:
    raise FileNotFoundError(
        f"No snapshot files found in {DATA_DIR.resolve()} for {MONTHS[0]}..{MONTHS[-1]}. "
        f"Check DATA_DIR and FILE_PREFIX."
    )

raw = pd.concat(frames, ignore_index=True)
del frames
print(f"\nLoaded {len(raw):,} total rows from {len(MONTHS)} months in {time.time() - t0:.1f}s")
print(f"Memory footprint: {raw.memory_usage(deep=True).sum() / 1e9:.3f} GB")

## 2. NAICS parsing — split "CODE|DESCRIPTION" and derive 2-6 digit rollups

Known dirty values from the source system (missing, '-1', 'UNKNOWN',
'******') are treated as invalid at every digit level rather than
silently truncated into a fake code.

In [ ]:
NAICS_SENTINELS = {"-1", "", "UNKNOWN", "******"}

def split_naics(raw_series: pd.Series) -> pd.DataFrame:
    s = raw_series.fillna("").astype(str)
    split = s.str.split("|", n=1, expand=True)
    code = split[0].str.strip()
    if split.shape[1] > 1:
        desc = split[1].str.strip()
    else:
        desc = pd.Series("", index=s.index)

    is_valid = code.str.fullmatch(r"\d{2,6}").fillna(False) & ~code.isin(NAICS_SENTINELS)

    out = pd.DataFrame({
        "naics_code": code.where(is_valid),
        "naics_desc": desc.where(is_valid),
        "naics_valid": is_valid,
    })
    for n in (2, 3, 4, 5, 6):
        long_enough = out["naics_code"].str.len() >= n
        out[f"naics{n}"] = out["naics_code"].str.slice(0, n).where(long_enough.fillna(False))
    return out

In [ ]:
src_naics = split_naics(raw["source_naics"]).add_prefix("source_")
dst_naics = split_naics(raw["dest_naics"]).add_prefix("dest_")

raw = pd.concat([raw.drop(columns=["source_naics", "dest_naics"]), src_naics, dst_naics], axis=1)

print("Source NAICS validity: {:.1%}".format(raw["source_naics_valid"].mean()))
print("Dest   NAICS validity: {:.1%}".format(raw["dest_naics_valid"].mean()))

## 3. Node attribute table (name + NAICS)

A node can appear as `source` in some rows and `dest` in others, and its
name/NAICS can drift slightly month to month (reclassification, minor
name changes). We union both sides and, per node, take the most recent
*non-null* value for each attribute (pandas groupby `.last()` skips NaNs
by default, so an early good value isn't lost to a later blank).

In [ ]:
def build_node_table(raw: pd.DataFrame) -> pd.DataFrame:
    # year_month is a row-level column (which month this transaction came
    # from); the naics/name columns are the ones split by source_/dest_.
    naics_cols = ["naics_code", "naics_desc", "naics_valid",
                  "naics2", "naics3", "naics4", "naics5", "naics6"]

    src = raw[["source", "source_name", "year_month"] + [f"source_{c}" for c in naics_cols]].copy()
    src.columns = ["node_id", "name", "year_month"] + naics_cols

    dst = raw[["dest", "dest_name", "year_month"] + [f"dest_{c}" for c in naics_cols]].copy()
    dst.columns = ["node_id", "name", "year_month"] + naics_cols

    both = pd.concat([src, dst], ignore_index=True).sort_values("year_month")
    nodes = both.groupby("node_id", sort=False).last().drop(columns="year_month").reset_index()
    return nodes

nodes = build_node_table(raw)
print(f"Unique nodes: {len(nodes):,}")

## 4. Aggregated edge table over the window

- amount_total : sum of monthly `amount` across the window
- volume_total : sum of monthly `volume` (txn count) across the window
- n_rels          : row count for the pair across the window (equals the
                 month count when snapshots have one row per pair per
                 month, which is the expected schema here)
- n_months_active : number of DISTINCT months the pair transacted -- the
                 true persistence signal (1..N_WINDOW_MONTHS). Use this,
                 not n_rels, for persistence logic; it won't overcount if
                 a month ever contains several rows for the same pair.

If this OOMs on the full history the way the monthly pipeline's
aggregate graph did, aggregate incrementally per month (running-sum a
dict keyed by (source,dest), discard each month's raw rows) instead of
concatenating everything first -- same principle as the streaming
design already used in the production pipeline.

In [ ]:
edges = (
    raw.groupby(["source", "dest"], sort=False)
       .agg(amount_total=("amount", "sum"),
            volume_total=("volume", "sum"),
            n_rels=("amount", "size"),
            n_months_active=("year_month", "nunique"))
       .reset_index()
)
edges["avg_amount_per_txn"] = edges["amount_total"] / edges["volume_total"].replace(0, np.nan)

n_touched = pd.unique(pd.concat([edges["source"], edges["dest"]], ignore_index=True))
print(f"Unique directed edges (source, dest pairs): {len(edges):,}")
print(f"Unique nodes touched by an edge:            {len(n_touched):,}")

### 4b. Persistence shape: consecutive streak and amount regularity

`n_months_active` alone can't tell a steady 8-month relationship from 8
scattered one-off months. Two extra signals separate them:

- `max_streak`  : longest run of CONSECUTIVE active months. A retainer or
                  standing panel relationship runs unbroken; opportunistic
                  one-offs don't.
- `amount_cv`   : coefficient of variation of the monthly amount. Steady
                  fee/retainer flows are LOW variance; subrogation recovery
                  remittances are lumpy (settlements close irregularly and
                  arrive net of contingency fees), so HIGH variance is
                  actually the more subrogation-like shape here.

Both are computed from a month-level edge table and merged onto `edges`.

In [ ]:
edge_month = (
    raw.groupby(["source", "dest", "year_month"], sort=False)
       .agg(amount=("amount", "sum"), volume=("volume", "sum"))
       .reset_index()
)
MONTH_IDX = {ym: i for i, ym in enumerate(MONTHS)}
edge_month["month_idx"] = edge_month["year_month"].map(MONTH_IDX)
edge_month = edge_month.sort_values(["source", "dest", "month_idx"], kind="mergesort")

# Longest consecutive-month run per pair, fully vectorized (no iterrows).
# A gap != 1 starts a new run; the first row of each pair yields NaN, which
# also compares != 1, so each pair correctly begins a fresh run.
gap = edge_month.groupby(["source", "dest"], sort=False)["month_idx"].diff()
edge_month["run_id"] = gap.ne(1).cumsum()
max_streak = (
    edge_month.groupby(["source", "dest", "run_id"], sort=False).size()
              .reset_index(name="run_len")
              .groupby(["source", "dest"], sort=False)["run_len"].max()
              .rename("max_streak").reset_index()
)

_amt = edge_month.groupby(["source", "dest"], sort=False)["amount"].agg(["mean", "std"])
_amt["amount_cv"] = _amt["std"] / _amt["mean"].replace(0, np.nan)
amount_cv = _amt[["amount_cv"]].reset_index()

edges = edges.merge(max_streak, on=["source", "dest"], how="left")
edges = edges.merge(amount_cv, on=["source", "dest"], how="left")
edges["max_streak"] = edges["max_streak"].fillna(1).astype(int)
edges["persistence_ratio"] = edges["n_months_active"] / N_WINDOW_MONTHS

print(f"max_streak      -- median {edges['max_streak'].median():.0f}, "
      f"max {edges['max_streak'].max():.0f}")
print(f"persistence_ratio -- median {edges['persistence_ratio'].median():.2f}")
print(f"amount_cv       -- median {edges['amount_cv'].median():.2f} "
      f"(NaN for single-month edges: {int(edges['amount_cv'].isna().sum()):,})")

## 5. Pick a graph backend based on actual scale

NetworkX is pure Python: great API, but construction and anything beyond
basic degree/neighbor queries gets uncomfortable somewhere in the low
hundreds of thousands of edges and painful well before a few million.
NetworKit is a C++/OpenMP backend built for exactly this scale, at the
cost of a clunkier API (integer node ids, fewer convenience methods).
Single-month snapshots already run 3-5M edges, so don't assume -- check
the actual aggregated count and branch on it.

In [ ]:
N_EDGE_THRESHOLD = 750_000
N_NODE_THRESHOLD = 750_000

n_edges = len(edges)
n_nodes = len(nodes)
use_networkit = (n_edges > N_EDGE_THRESHOLD) or (n_nodes > N_NODE_THRESHOLD)

print(f"nodes={n_nodes:,}  edges={n_edges:,}  -> backend = "
      f"{'networkit' if use_networkit else 'networkx'}")

In [ ]:
idx_of: Optional[Dict[str, int]] = None
id_of: Optional[Dict[int, str]] = None

if use_networkit:
    try:
        import networkit as nk
    except ImportError:
        raise ImportError(
            "This graph needs networkit at this scale. Install with: "
            "pip install networkit --break-system-packages"
        )

    node_ids = pd.unique(pd.concat([edges["source"], edges["dest"]], ignore_index=True))
    idx_of = {nid: i for i, nid in enumerate(node_ids)}
    id_of = {i: nid for nid, i in idx_of.items()}

    G = nk.Graph(n=len(node_ids), weighted=True, directed=True)
    for row in edges.itertuples(index=False):
        G.addEdge(idx_of[row.source], idx_of[row.dest], row.amount_total)

    edge_attr = edges.set_index(["source", "dest"])

else:
    import networkx as nx

    G = nx.DiGraph()
    G.add_nodes_from(nodes["node_id"])
    G.add_edges_from(
        (r.source, r.dest, {"amount": r.amount_total,
                             "volume": r.volume_total,
                             "n_rels": r.n_rels,
                             "avg_amount_per_txn": r.avg_amount_per_txn})
        for r in edges.itertuples(index=False)
    )
    node_attr_cols = ["name", "naics_code", "naics_desc", "naics2", "naics3", "naics4", "naics5", "naics6"]
    nx.set_node_attributes(G, nodes.set_index("node_id")[node_attr_cols].to_dict("index"))

n_g_nodes = G.numberOfNodes() if use_networkit else G.number_of_nodes()
n_g_edges = G.numberOfEdges() if use_networkit else G.number_of_edges()
print(f"Graph built: {n_g_nodes:,} nodes, {n_g_edges:,} edges")

## 6. EDA -- surface candidate NAICS codes for "insurance / subrogation"

Rather than hardcoding NAICS codes from memory, pull the actual distinct
(code, description) pairs in *your* data whose description mentions
insurance/claims/subrogation-adjacent terms, so codes are picked off
what's really present.

In [ ]:
KEYWORD_PATTERN = r"insur|casualt|reinsur|subrogat|claim|underwrit|assurance"

candidate_naics = (
    nodes.loc[nodes["naics_desc"].str.contains(KEYWORD_PATTERN, case=False, na=False),
              ["naics_code", "naics_desc"]]
    .drop_duplicates()
    .sort_values("naics_code")
)
print(candidate_naics.to_string(index=False))

## 7. Node classification: insurer / law firm / other

Tags every node with a `node_class` so the rest of the notebook can reason
about insurer<->law-firm flows directly. Priority: an insurer NAICS/name
wins over law-firm signals on conflicts (524 is the more specific signal
for our purpose). `has_trust_flag` separately marks attorney/plaintiff-firm
trust (IOLTA) accounts -- the cleanest single marker of settlement-money
movement, and it still fires when NAICS is missing.

In [ ]:
# Surface the legal-services NAICS codes actually present (mirrors the
# insurance EDA in section 6), so law-firm coverage can be sanity-checked.
LEGAL_KEYWORD_PATTERN = r"legal|attorney|lawyer|\blaw\b|counsel"
candidate_legal_naics = (
    nodes.loc[nodes["naics_desc"].str.contains(LEGAL_KEYWORD_PATTERN, case=False, na=False),
              ["naics_code", "naics_desc"]]
    .drop_duplicates()
    .sort_values("naics_code")
)
print("Legal-services NAICS codes present in the data:")
print(candidate_legal_naics.to_string(index=False) if len(candidate_legal_naics)
      else "  (none by description -- relying on NAICS 5411 prefix + name patterns)")

In [ ]:
def classify_nodes(nodes: pd.DataFrame) -> pd.DataFrame:
    name = nodes["name"].fillna("")
    naics3 = nodes["naics3"].fillna("")
    naics4 = nodes["naics4"].fillna("")

    is_insurer = (
        naics3.isin(INSURER_NAICS3)
        | name.str.contains(INSURER_NAME_PATTERN, case=False, regex=True, na=False)
    )
    is_trust = name.str.contains(TRUST_ACCOUNT_PATTERN, case=False, regex=True, na=False)
    legal_naics = naics4.isin(LAW_FIRM_NAICS4)
    strong_name = name.str.contains(LAW_FIRM_NAME_PATTERN, case=False, regex=True, na=False)

    # A VALID NAICS that is clearly not legal services outranks a name hint.
    # Without this, "<something> LEGAL SERVICES" style names coded to an
    # unrelated industry, or shared entity suffixes, leak into the class.
    # Note this reads `naics_observed` only -- never an imputed value.
    contradicting_naics = nodes["naics_valid"].fillna(False) & ~legal_naics

    is_law_firm = (
        legal_naics
        | (strong_name & ~contradicting_naics)
        | is_trust                      # IOLTA/client-trust wording is decisive on its own
    ) & ~is_insurer                     # insurer signal takes precedence on conflicts

    out = nodes.copy()
    out["node_class"] = np.select([is_insurer, is_law_firm],
                                  ["insurer", "law_firm"], default="other")
    out["has_trust_flag"] = is_trust & ~is_insurer
    # how each law firm was caught -- exposes NAICS gaps vs. name-only hits
    out["law_firm_by_naics"] = legal_naics & (out["node_class"] == "law_firm")
    out["law_firm_by_name_only"] = (out["node_class"] == "law_firm") & ~out["law_firm_by_naics"]
    # audit trail: name looked legal but a valid non-legal NAICS overrode it
    out["law_name_rejected_by_naics"] = strong_name & contradicting_naics & ~is_insurer
    return out

nodes = classify_nodes(nodes)

print("\nNode class counts:")
print(nodes["node_class"].value_counts().to_string())
print(f"\nLaw firms: {int((nodes['node_class']=='law_firm').sum()):,}  "
      f"(trust-flagged: {int(nodes['has_trust_flag'].sum()):,}; "
      f"name-only / NAICS missing: {int(nodes['law_firm_by_name_only'].sum()):,})")
print("\nSample law firms detected:")
print(nodes.loc[nodes["node_class"] == "law_firm",
                ["node_id", "name", "naics_code", "has_trust_flag", "law_firm_by_name_only"]]
      .head(15).to_string(index=False))

## 8. Node selection: NAICS prefix list + optional name regex (insurers)

In [ ]:
def select_nodes(nodes: pd.DataFrame,
                  naics_prefixes: Optional[List[str]] = None,
                  naics_level: int = 3,
                  name_regex: Optional[str] = None) -> pd.DataFrame:
    """
    naics_prefixes : e.g. ['524'] to match the naics3 column, or specific
                      6-digit codes with naics_level=6.
    name_regex     : optional, applied to `name` (case-insensitive), OR'd
                      with the NAICS match if both are given.
    """
    mask = pd.Series(False, index=nodes.index)
    if naics_prefixes:
        col = f"naics{naics_level}"
        mask = mask | nodes[col].isin(naics_prefixes)
    if name_regex:
        mask = mask | nodes["name"].str.contains(name_regex, case=False, na=False, regex=True)
    return nodes.loc[mask].copy()

# Adjust this once you've looked at `candidate_naics` above.
# 524 = Insurance Carriers & Related Activities (carriers, TPAs, claims adjusters, brokers)
seed_nodes = select_nodes(
    nodes,
    naics_prefixes=["524"],
    naics_level=3,
    name_regex=r"subrogat|arbitration forum",
)
print(f"\nSeed candidates: {len(seed_nodes):,}")
print(seed_nodes.head(20).to_string(index=False))

## 9. First-order ego graph around the seed set (both payers and payees)

This is the union of in- and out-neighbors across the whole seed set,
then the induced subgraph on {seeds} u neighbors -- not a single-center
nx.ego_graph, since we want edges between the seeds' counterparties too
(e.g. two suspected subrogation partners that also share a counterparty).

In [ ]:
def ego_subgraph(G, seed_ids, use_networkit=False, idx_of=None, id_of=None):
    if use_networkit:
        seed_idx = [idx_of[s] for s in seed_ids if s in idx_of]
        keep = set(seed_idx)
        for s in seed_idx:
            keep.update(G.iterNeighbors(s))
            keep.update(G.iterInNeighbors(s))
        sub = nk.graphtools.subgraphFromNodes(G, keep)
        kept_ids = [id_of[i] for i in keep]
        return sub, kept_ids
    else:
        keep = set(seed_ids)
        for s in seed_ids:
            if G.has_node(s):
                keep.update(G.successors(s))
                keep.update(G.predecessors(s))
        sub = G.subgraph(keep).copy()
        return sub, list(keep)

seed_ids = seed_nodes["node_id"].tolist()
if not seed_ids:
    raise ValueError(
        "No seed nodes matched -- widen naics_prefixes/name_regex in select_nodes(), "
        "or check candidate_naics above for the right codes."
    )

if use_networkit:
    sub, kept_ids = ego_subgraph(G, seed_ids, use_networkit=True, idx_of=idx_of, id_of=id_of)
    print(f"Ego subgraph: {sub.numberOfNodes():,} nodes, {sub.numberOfEdges():,} edges")
else:
    sub, kept_ids = ego_subgraph(G, seed_ids, use_networkit=False)
    print(f"Ego subgraph: {sub.number_of_nodes():,} nodes, {sub.number_of_edges():,} edges")

## 10. Payer / payee profile + reciprocal-flow check

The subrogation "signature" in an aggregated window is a pair of nodes
with edges running BOTH ways: money moving in both directions between
the same two institutions, unusual for a typical vendor/customer
relationship but exactly what you'd expect from two carriers that each
subrogate against the other across a portfolio of claims. This is
checked on the *aggregated* graph deliberately -- subrogation recovery
lags the original claim payment by weeks to months, so same-month
reciprocity would badly undercount real relationships.

In [ ]:
def payer_payee_profile(edges: pd.DataFrame, node_id: str) -> dict:
    incoming = edges.loc[edges["dest"] == node_id]
    outgoing = edges.loc[edges["source"] == node_id]
    return {
        "node_id": node_id,
        "n_payers": incoming["source"].nunique(),
        "n_payees": outgoing["dest"].nunique(),
        "amount_in": incoming["amount_total"].sum(),
        "amount_out": outgoing["amount_total"].sum(),
        "volume_in": incoming["volume_total"].sum(),
        "volume_out": outgoing["volume_total"].sum(),
    }

profiles = pd.DataFrame([payer_payee_profile(edges, n) for n in seed_ids])
profiles = profiles.merge(nodes[["node_id", "name"]], on="node_id", how="left")
print(profiles.sort_values("amount_in", ascending=False).head(20).to_string(index=False))

In [ ]:
def reciprocal_pairs(edges: pd.DataFrame, node_subset: Set[str]) -> pd.DataFrame:
    e = edges.loc[edges["source"].isin(node_subset) & edges["dest"].isin(node_subset),
                  ["source", "dest", "amount_total", "volume_total", "n_rels"]]
    merged = e.merge(e, left_on=["source", "dest"], right_on=["dest", "source"],
                      suffixes=("_fwd", "_rev"))
    merged = merged.loc[merged["source_fwd"] < merged["dest_fwd"]]
    merged["balance_ratio"] = (
        merged[["amount_total_fwd", "amount_total_rev"]].min(axis=1) /
        merged[["amount_total_fwd", "amount_total_rev"]].max(axis=1)
    )
    return merged.sort_values("balance_ratio", ascending=False)

recip = reciprocal_pairs(edges, set(seed_ids))
name_lookup = nodes[["node_id", "name"]]
recip = recip.merge(name_lookup.rename(columns={"node_id": "source_fwd", "name": "source_name"}),
                     on="source_fwd", how="left")
recip = recip.merge(name_lookup.rename(columns={"node_id": "dest_fwd", "name": "dest_name"}),
                     on="dest_fwd", how="left")

print(f"\nReciprocal pairs among seed nodes: {len(recip):,}")
cols = ["source_name", "dest_name", "amount_total_fwd", "amount_total_rev",
        "balance_ratio", "n_rels_fwd", "n_rels_rev"]
print(recip[cols].head(20).to_string(index=False))

## 11. Reinsurance vs. subrogation discriminator

Carrier-to-carrier reciprocal flow is *also* what reinsurance treaty
settlement looks like structurally. The cleanest separator is the average
amount per transaction: subrogation = many moderate claim settlements;
reinsurance = few large treaty movements. Inspect the distribution across
the seed set's edges before trusting any single pair as "subrogation".

In [ ]:
seed_set = set(seed_ids)
seed_edges = edges.loc[edges["source"].isin(seed_set) & edges["dest"].isin(seed_set)].copy()
if len(seed_edges):
    q = seed_edges["avg_amount_per_txn"].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99])
    print("avg_amount_per_txn distribution across seed-to-seed edges:")
    print(q.to_string())
    print(f"\nmedian volume per edge: {seed_edges['volume_total'].median():,.0f}")
    print("High avg-amount + low volume edges (reinsurance-like, candidates to EXCLUDE):")
    reins_like = seed_edges.sort_values("avg_amount_per_txn", ascending=False).head(10)
    reins_like = reins_like.merge(name_lookup.rename(columns={"node_id": "source", "name": "s_name"}), on="source", how="left")
    reins_like = reins_like.merge(name_lookup.rename(columns={"node_id": "dest", "name": "d_name"}), on="dest", how="left")
    print(reins_like[["s_name", "d_name", "amount_total", "volume_total", "avg_amount_per_txn"]].to_string(index=False))
else:
    print("No seed-to-seed edges to profile.")

## 12. Insurance <-> law-firm persistent transactions

For workers' comp and general liability, recoveries usually do NOT move
carrier->carrier -- they move through plaintiff-firm and attorney trust
accounts. So a persistent insurer<->law-firm edge is often the ONLY
structural trace those recoveries leave, and the carrier-to-carrier
reciprocity check (section 10) misses them entirely. Here we isolate every
insurer<->law-firm edge, attach persistence, and tier it. Direction is
informative: firm->insurer is the recovery inflow (settlement proceeds
reaching the recovering carrier); insurer->firm is defense / recovery-
vendor / settlement funding.

In [ ]:
node_class_of = nodes.set_index("node_id")["node_class"]
trust_of = nodes.set_index("node_id")["has_trust_flag"]

edges["source_class"] = edges["source"].map(node_class_of)
edges["dest_class"] = edges["dest"].map(node_class_of)

ins_law = edges.loc[
    ((edges["source_class"] == "insurer") & (edges["dest_class"] == "law_firm")) |
    ((edges["source_class"] == "law_firm") & (edges["dest_class"] == "insurer"))
].copy()

ins_law["direction"] = np.where(ins_law["source_class"] == "insurer",
                                 "insurer_to_firm", "firm_to_insurer")
ins_law["insurer_id"] = np.where(ins_law["source_class"] == "insurer",
                                  ins_law["source"], ins_law["dest"])
ins_law["firm_id"] = np.where(ins_law["source_class"] == "law_firm",
                               ins_law["source"], ins_law["dest"])
ins_law["is_trust_firm"] = ins_law["firm_id"].map(trust_of).fillna(False).astype(bool)
ins_law["persistence_ratio"] = ins_law["n_months_active"] / N_WINDOW_MONTHS


def tier_ins_law(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    persist = d["n_months_active"]
    trust = d["is_trust_firm"].astype(bool)
    very_persistent = persist >= int(np.ceil(0.8 * N_WINDOW_MONTHS))
    conditions = [
        ((persist >= PERSISTENCE_MIN_MONTHS) & trust) | very_persistent,   # A
        (persist >= PERSISTENCE_MIN_MONTHS) | ((persist >= 3) & trust),    # B
    ]
    d["confidence_tier"] = np.select(conditions, ["A_high", "B_medium"], default="C_low")
    return d


ins_law = tier_ins_law(ins_law)
ins_law = ins_law.merge(name_lookup.rename(columns={"node_id": "insurer_id", "name": "insurer_name"}),
                        on="insurer_id", how="left")
ins_law = ins_law.merge(name_lookup.rename(columns={"node_id": "firm_id", "name": "firm_name"}),
                        on="firm_id", how="left")

ins_law_persistent = ins_law.loc[ins_law["n_months_active"] >= PERSISTENCE_MIN_MONTHS]

# how many law firms sit inside the seed insurers' ego subgraph (section 9)?
law_firm_ids = set(nodes.loc[nodes["node_class"] == "law_firm", "node_id"])
law_in_ego = law_firm_ids.intersection(set(kept_ids))

print(f"Insurer<->law-firm edges: {len(ins_law):,}  "
      f"(persistent >= {PERSISTENCE_MIN_MONTHS} months: {len(ins_law_persistent):,})")
print(f"Law-firm nodes inside seed insurers' ego subgraph: {len(law_in_ego):,}")
print("\nTier counts (all insurer<->firm edges):")
print(ins_law["confidence_tier"].value_counts().to_string() if len(ins_law) else "  (none)")
print("\nTop persistent insurer<->law-firm relationships:")
_show = ["insurer_name", "firm_name", "direction", "n_months_active",
         "volume_total", "amount_total", "is_trust_firm", "confidence_tier"]
print(ins_law.sort_values(["n_months_active", "volume_total"], ascending=False)
      [_show].head(25).to_string(index=False) if len(ins_law)
      else "  (no insurer<->law-firm edges -- check classification in section 7)")

## 12b. Pair-level directional analysis — the part that matters

Section 12 tiers each DIRECTED edge on persistence, which is not enough:
a standing panel-defense-counsel relationship (carrier pays the same firm
every single month) is maximally persistent and will out-rank real
recoveries if direction is ignored. Carrier<->carrier subrogation shows up
as BALANCED reciprocity (section 10); carrier<->law-firm does not. Here
direction IS the signal:

| Pattern | Reading |
|---|---|
| **firm -> insurer dominant** | Plaintiff/recovery firm trust account remitting settlement proceeds to satisfy a WC lien or subrogation interest. **The recovery signal.** |
| **insurer -> firm dominant** | Panel defense counsel / litigation funding. Highly persistent by nature — the main confound, NOT a find. |
| **bidirectional** | Recovery counsel: carrier funds pursuit, firm remits recoveries net of contingency fee. Genuinely interesting. |

So we collapse the two directed edges into one insurer<->firm pair, measure
which way the money actually leans, and let that gate the tier. A
defense-dominant pair can never reach A/B no matter how persistent it is.

In [ ]:
_PAIR_COLS = ["insurer_id", "firm_id", "amount_total", "volume_total",
              "n_months_active", "max_streak", "amount_cv"]

def build_ins_law_pairs(ins_law: pd.DataFrame) -> pd.DataFrame:
    """Collapse directed insurer<->law-firm edges into one row per pair."""
    f2i = (ins_law.loc[ins_law["direction"] == "firm_to_insurer", _PAIR_COLS]
           .rename(columns={c: f"{c}_f2i" for c in _PAIR_COLS[2:]}))
    i2f = (ins_law.loc[ins_law["direction"] == "insurer_to_firm", _PAIR_COLS]
           .rename(columns={c: f"{c}_i2f" for c in _PAIR_COLS[2:]}))

    pairs = f2i.merge(i2f, on=["insurer_id", "firm_id"], how="outer")

    for side in ("f2i", "i2f"):
        for c in ("amount_total", "volume_total", "n_months_active", "max_streak"):
            pairs[f"{c}_{side}"] = pairs[f"{c}_{side}"].fillna(0)

    amt_f2i = pairs["amount_total_f2i"]
    amt_i2f = pairs["amount_total_i2f"]
    total = (amt_f2i + amt_i2f).replace(0, np.nan)
    # share of the pair's money flowing FIRM -> INSURER (1.0 = pure recovery
    # inflow, 0.0 = pure outbound defense/funding)
    pairs["recovery_ratio"] = amt_f2i / total

    pairs["months_active_max"] = pairs[["n_months_active_f2i", "n_months_active_i2f"]].max(axis=1)
    pairs["max_streak_max"] = pairs[["max_streak_f2i", "max_streak_i2f"]].max(axis=1)
    pairs["amount_total_both"] = amt_f2i + amt_i2f
    pairs["volume_total_both"] = pairs["volume_total_f2i"] + pairs["volume_total_i2f"]

    rr = pairs["recovery_ratio"]
    pairs["flow_pattern"] = np.select(
        [rr >= RECOVERY_RATIO_HIGH, rr <= RECOVERY_RATIO_LOW],
        ["FIRM_TO_INSURER_DOMINANT", "INSURER_TO_FIRM_DOMINANT"],
        default="BIDIRECTIONAL",
    )
    pairs["is_trust_firm"] = pairs["firm_id"].map(trust_of).fillna(False).astype(bool)
    return pairs


def tier_ins_law_pairs(pairs: pd.DataFrame) -> pd.DataFrame:
    """
    Direction-aware tiering. Defense-dominant pairs are labelled and pushed
    out of the candidate set FIRST, so no amount of persistence can promote
    them -- the specific failure mode of scoring on persistence alone.
    """
    p = pairs.copy()
    is_recovery = p["flow_pattern"].isin(["FIRM_TO_INSURER_DOMINANT", "BIDIRECTIONAL"])
    persistent = p["months_active_max"] >= PERSISTENCE_MIN_MONTHS
    strong_streak = p["max_streak_max"] >= int(np.ceil(0.7 * N_WINDOW_MONTHS))
    trust = p["is_trust_firm"].astype(bool)

    conditions = [
        (~is_recovery) & persistent,                       # defense panel: excluded
        is_recovery & persistent & (trust | strong_streak),  # A
        is_recovery & persistent,                          # B
        is_recovery,                                       # C (recovery-shaped, sparse)
    ]
    choices = ["X_defense_panel", "A_high", "B_medium", "C_low"]
    p["confidence_tier"] = np.select(conditions, choices, default="D_weak")
    return p.sort_values(["confidence_tier", "months_active_max", "amount_total_both"],
                          ascending=[True, False, False])


ins_law_pairs = build_ins_law_pairs(ins_law)
ins_law_pairs = tier_ins_law_pairs(ins_law_pairs)
ins_law_pairs = ins_law_pairs.merge(
    name_lookup.rename(columns={"node_id": "insurer_id", "name": "insurer_name"}),
    on="insurer_id", how="left")
ins_law_pairs = ins_law_pairs.merge(
    name_lookup.rename(columns={"node_id": "firm_id", "name": "firm_name"}),
    on="firm_id", how="left")

print(f"Insurer<->law-firm PAIRS: {len(ins_law_pairs):,}\n")
print("Flow pattern:")
print(ins_law_pairs["flow_pattern"].value_counts().to_string())
print("\nConfidence tier (direction-aware):")
print(ins_law_pairs["confidence_tier"].value_counts().to_string())

_pshow = ["insurer_name", "firm_name", "flow_pattern", "recovery_ratio",
          "months_active_max", "max_streak_max", "amount_total_both",
          "is_trust_firm", "confidence_tier"]
print("\n--- Subrogation recovery candidates (A/B) ---")
_cand = ins_law_pairs.loc[ins_law_pairs["confidence_tier"].isin(["A_high", "B_medium"])]
print(_cand[_pshow].head(25).to_string(index=False) if len(_cand) else "  (none)")

print("\n--- Excluded as defense panel (persistent but wrong direction) ---")
_def = ins_law_pairs.loc[ins_law_pairs["confidence_tier"] == "X_defense_panel"]
print(_def[_pshow].head(15).to_string(index=False) if len(_def) else "  (none)")

## 13. Law-firm recovery hubs (firms bridging many insurers)

A law firm transacting with MANY insurers is the connective tissue of the
recovery network -- a subrogation-recovery specialist or settlement
administrator, the legal-layer analogue of the clearinghouse hub. Ranked by
distinct insurers reached, then volume. `amount_from_insurers` (money the
firm receives) vs `amount_to_insurers` (recoveries it remits) splits the
funding role from the recovery role.

In [ ]:
if len(ins_law):
    firm_hub = (
        ins_law.groupby("firm_id")
        .agg(n_insurers=("insurer_id", "nunique"),
             n_edges=("insurer_id", "size"),
             max_months_active=("n_months_active", "max"),
             total_volume=("volume_total", "sum"),
             total_amount=("amount_total", "sum"),
             is_trust_firm=("is_trust_firm", "max"))
        .reset_index()
    )
    recv = (ins_law.loc[ins_law["direction"] == "insurer_to_firm"]
            .groupby("firm_id")["amount_total"].sum().rename("amount_from_insurers"))
    paid = (ins_law.loc[ins_law["direction"] == "firm_to_insurer"]
            .groupby("firm_id")["amount_total"].sum().rename("amount_to_insurers"))
    firm_hub = (firm_hub.merge(recv, on="firm_id", how="left")
                        .merge(paid, on="firm_id", how="left"))
    firm_hub[["amount_from_insurers", "amount_to_insurers"]] = \
        firm_hub[["amount_from_insurers", "amount_to_insurers"]].fillna(0.0)
    firm_hub = firm_hub.merge(name_lookup.rename(columns={"node_id": "firm_id", "name": "firm_name"}),
                              on="firm_id", how="left")
    firm_hub = firm_hub.sort_values(["n_insurers", "total_volume"], ascending=False)

    print(f"Distinct law firms in insurer flows: {len(firm_hub):,}")
    print("\nTop law-firm recovery hubs:")
    _cols = ["firm_name", "n_insurers", "max_months_active", "total_volume",
             "amount_from_insurers", "amount_to_insurers", "is_trust_firm"]
    print(firm_hub[_cols].head(20).to_string(index=False))
else:
    firm_hub = pd.DataFrame()
    print("No insurer<->law-firm edges found -- check classification thresholds in section 7.")

## 14. Per-insurer law-firm exposure

For each insurer: how many law firms it deals with, how many PERSISTENTLY,
and the NET direction of money. Persistent NET INFLOW from law firms means
the carrier is capturing recoveries (a strong subrogation-recovery signal);
net OUTFLOW suggests it is funding defense / litigation / recovery vendors.

In [ ]:
if len(ins_law):
    base = (ins_law.groupby("insurer_id")
            .agg(n_firms=("firm_id", "nunique"),
                 max_months_active=("n_months_active", "max"),
                 total_volume=("volume_total", "sum")).reset_index())
    persist_firms = (ins_law.loc[ins_law["n_months_active"] >= PERSISTENCE_MIN_MONTHS]
                     .groupby("insurer_id")["firm_id"].nunique().rename("n_persistent_firms"))
    inflow = (ins_law.loc[ins_law["direction"] == "firm_to_insurer"]
              .groupby("insurer_id")["amount_total"].sum().rename("amount_from_firms"))
    outflow = (ins_law.loc[ins_law["direction"] == "insurer_to_firm"]
               .groupby("insurer_id")["amount_total"].sum().rename("amount_to_firms"))
    insurer_exposure = (base
        .merge(persist_firms, on="insurer_id", how="left")
        .merge(inflow, on="insurer_id", how="left")
        .merge(outflow, on="insurer_id", how="left"))
    _fill = ["n_persistent_firms", "amount_from_firms", "amount_to_firms"]
    insurer_exposure[_fill] = insurer_exposure[_fill].fillna(0)
    insurer_exposure["net_from_firms"] = (insurer_exposure["amount_from_firms"]
                                          - insurer_exposure["amount_to_firms"])
    insurer_exposure = insurer_exposure.merge(
        name_lookup.rename(columns={"node_id": "insurer_id", "name": "insurer_name"}),
        on="insurer_id", how="left")
    insurer_exposure = insurer_exposure.sort_values(
        ["n_persistent_firms", "net_from_firms"], ascending=False)

    print("Per-insurer law-firm exposure (net_from_firms > 0 => recovery inflow):")
    _cols = ["insurer_name", "n_firms", "n_persistent_firms", "amount_from_firms",
             "amount_to_firms", "net_from_firms"]
    print(insurer_exposure[_cols].head(20).to_string(index=False))
else:
    insurer_exposure = pd.DataFrame()
    print("No insurer<->law-firm edges to profile.")

## 15. Confidence tiering (carrier-to-carrier reciprocal pairs)

Since there is no transaction-level ground truth yet, every candidate ships
with an explicit confidence tier rather than a binary label. Tiers here are
structural heuristics -- tune the thresholds against your real distributions
and, once available, against AF / claims / addenda labels. (Insurer<->law-
firm relationships are tiered separately in section 12.)

In [ ]:
def tier_reciprocal_pairs(recip: pd.DataFrame) -> pd.DataFrame:
    r = recip.copy()
    bal = r["balance_ratio"].fillna(0)
    persist = r[["n_rels_fwd", "n_rels_rev"]].min(axis=1)
    conditions = [
        (bal >= 0.6) & (persist >= 6),   # balanced AND recurring across many months
        (bal >= 0.4) & (persist >= 3),   # moderately balanced, some recurrence
    ]
    choices = ["A_high", "B_medium"]
    r["confidence_tier"] = np.select(conditions, choices, default="C_low")
    return r.sort_values(["confidence_tier", "balance_ratio"], ascending=[True, False])

recip_tiered = tier_reciprocal_pairs(recip)
print(recip_tiered["confidence_tier"].value_counts().to_string())
print()
print(recip_tiered[["source_name", "dest_name", "balance_ratio",
                    "n_rels_fwd", "n_rels_rev", "confidence_tier"]].head(20).to_string(index=False))

## 16. Persist candidate tables

Save every artifact of this pass -- classified entities, seed insurers,
per-seed profiles, tiered carrier reciprocal pairs, and the new law-firm
tables (tiered insurer<->firm edges, recovery hubs, per-insurer exposure) --
so a closer look can continue in a later session without re-aggregating.

In [ ]:
OUT_DIR = Path("./subrogation_candidates")
OUT_DIR.mkdir(exist_ok=True)

tag = f"{START_YM}_{END_YM}"
seed_out = seed_nodes[["node_id", "name", "naics_code", "naics_desc"]]
seed_out.to_csv(OUT_DIR / f"seed_entities_{tag}.csv", index=False)
profiles.to_csv(OUT_DIR / f"seed_profiles_{tag}.csv", index=False)
recip_tiered.to_csv(OUT_DIR / f"reciprocal_pairs_tiered_{tag}.csv", index=False)

# classified entities (insurers + law firms) and the law-firm tables
nodes.loc[nodes["node_class"] != "other",
          ["node_id", "name", "naics_code", "node_class", "has_trust_flag",
           "law_firm_by_name_only"]].to_csv(OUT_DIR / f"classified_entities_{tag}.csv", index=False)
if len(ins_law):
    (ins_law.sort_values(["confidence_tier", "n_months_active"], ascending=[True, False])
     .to_csv(OUT_DIR / f"insurer_lawfirm_edges_tiered_{tag}.csv", index=False))
if len(firm_hub):
    firm_hub.to_csv(OUT_DIR / f"lawfirm_recovery_hubs_{tag}.csv", index=False)
if len(insurer_exposure):
    insurer_exposure.to_csv(OUT_DIR / f"insurer_lawfirm_exposure_{tag}.csv", index=False)

print("Wrote:")
for p in sorted(OUT_DIR.glob("*.csv")):
    print(f"  {p}  ({p.stat().st_size:,} bytes)")

## 17. Visualizing the subrogation subgraph

A generic spring layout of this subgraph is a hairball and tells you
nothing -- and per the roadmap, hub nodes (~100K degree) break the
visualisation outright. Three design choices make it readable:

1. **Type the nodes.** Colour by `node_class` (insurer / law firm / other),
   with trust accounts outlined -- structure only becomes legible once you
   can see what kind of entity each node is.
2. **Make direction mean something.** Firm->insurer edges (the recovery
   signal) are drawn in green; insurer->firm (defense/funding) in muted
   red; insurer<->insurer in blue. The picture then reads as a story about
   money direction rather than a ball of arrows.
3. **Cap the size.** Always take a top-N-by-weight subgraph and drop
   nodes above a degree cap, so a clearinghouse hub can't swamp the layout.

Two renderers: a static matplotlib view (slides, docs) and an interactive
pyvis HTML view (exploration, drag/zoom/hover). For the Streamlit hand-off
the same typed-node/typed-edge model maps onto st-link-analysis.

In [ ]:
import math

PALETTE = {
    "insurer":  "#2E5EAA",   # blue
    "law_firm": "#E08A1E",   # orange
    "other":    "#B8BDC4",   # grey
}
EDGE_COLORS = {
    "firm_to_insurer": "#2E8B57",   # green  -- recovery inflow (the signal)
    "insurer_to_firm": "#C0504D",   # red    -- defense / funding (confound)
    "insurer_to_insurer": "#4472C4",  # blue -- carrier<->carrier subrogation
    "other": "#CCCCCC",
}


def edge_kind(u_class: str, v_class: str) -> str:
    if u_class == "law_firm" and v_class == "insurer":
        return "firm_to_insurer"
    if u_class == "insurer" and v_class == "law_firm":
        return "insurer_to_firm"
    if u_class == "insurer" and v_class == "insurer":
        return "insurer_to_insurer"
    return "other"


def build_viz_graph(edges: pd.DataFrame,
                     nodes: pd.DataFrame,
                     node_ids: Set[str],
                     top_n_edges: int = 150,
                     max_degree=None,
                     weight_col: str = "amount_total",
                     include_kinds: Optional[Set[str]] = None):
    """
    Induced subgraph on `node_ids`, trimmed to the top `top_n_edges` by
    `weight_col`. `max_degree` drops hub nodes before layout (the documented
    failure mode); pass None to keep everything. `include_kinds` restricts to
    given edge semantics, e.g. {"firm_to_insurer", "insurer_to_firm"} to
    isolate the law-firm story from carrier<->carrier noise.
    """
    import networkx as nx

    sub_e = edges.loc[edges["source"].isin(node_ids) & edges["dest"].isin(node_ids)].copy()

    if include_kinds is not None and len(sub_e):
        _cls = nodes.set_index("node_id")["node_class"]
        kinds = [edge_kind(_cls.get(s, "other"), _cls.get(d, "other"))
                 for s, d in zip(sub_e["source"], sub_e["dest"])]
        sub_e = sub_e.loc[pd.Series(kinds, index=sub_e.index).isin(include_kinds)]
        print(f"  filtered to edge kinds {sorted(include_kinds)}: {len(sub_e):,} edges")

    if max_degree is not None and len(sub_e):
        deg = pd.concat([sub_e["source"], sub_e["dest"]]).value_counts()
        n_sub_nodes = deg.size
        if n_sub_nodes < HUB_SUPPRESS_MIN_NODES:
            # On a small curated candidate set every node looks like a hub --
            # the concept only means something once the degree distribution
            # is skewed. Skip rather than gut the picture.
            print(f"  subgraph has {n_sub_nodes} nodes (< {HUB_SUPPRESS_MIN_NODES}); "
                  f"skipping hub suppression")
        else:
            # float in (0,1] -> QUANTILE of the degree distribution (scale-free);
            # int -> absolute degree cap
            cap = (deg.quantile(max_degree)
                   if isinstance(max_degree, float) and 0 < max_degree <= 1
                   else max_degree)
            hubs = set(deg[deg > cap].index)
            if hubs:
                hub_names = [str(nodes.loc[nodes["node_id"] == h, "name"].squeeze())
                             for h in list(hubs)[:5]]
                print(f"  dropping {len(hubs)} hub node(s) with degree > {cap:.0f} "
                      f"(of {n_sub_nodes} nodes): {hub_names}"
                      f"{' ...' if len(hubs) > 5 else ''}")
                trimmed = sub_e.loc[~sub_e["source"].isin(hubs) & ~sub_e["dest"].isin(hubs)]
                if trimmed.empty:
                    print("  WARNING: hub cap removed every edge -- keeping unsuppressed "
                          "view. Raise HUB_DEGREE_CAP or set it to None.")
                else:
                    sub_e = trimmed

    if len(sub_e) > top_n_edges:
        print(f"  trimming {len(sub_e):,} edges -> top {top_n_edges} by {weight_col}")
        sub_e = sub_e.nlargest(top_n_edges, weight_col)

    cls = nodes.set_index("node_id")["node_class"]
    nm = nodes.set_index("node_id")["name"]
    trust = nodes.set_index("node_id")["has_trust_flag"]

    g = nx.DiGraph()
    for r in sub_e.itertuples(index=False):
        uc, vc = cls.get(r.source, "other"), cls.get(r.dest, "other")
        g.add_node(r.source, node_class=uc, name=nm.get(r.source, r.source),
                   trust=bool(trust.get(r.source, False)))
        g.add_node(r.dest, node_class=vc, name=nm.get(r.dest, r.dest),
                   trust=bool(trust.get(r.dest, False)))
        g.add_edge(r.source, r.dest,
                   weight=float(getattr(r, weight_col)),
                   volume=float(r.volume_total),
                   months=int(r.n_months_active),
                   kind=edge_kind(uc, vc))
    return g

### 17a. Static view (matplotlib)

Bipartite layout puts insurers on the left and law firms on the right, so
the recovery direction reads left-to-right instead of tangling. Falls back
to a spring layout for mixed/carrier-only subgraphs.

In [ ]:
def draw_static(g, title="Subrogation subgraph", bipartite=True, figsize=(15, 10)):
    import matplotlib.pyplot as plt
    import networkx as nx

    if g.number_of_nodes() == 0:
        print("Empty graph -- nothing to draw.")
        return None, None

    insurers = [n for n, d in g.nodes(data=True) if d["node_class"] == "insurer"]
    firms = [n for n, d in g.nodes(data=True) if d["node_class"] == "law_firm"]
    others = [n for n, d in g.nodes(data=True) if d["node_class"] == "other"]

    if bipartite and insurers and firms:
        pos = {}
        for i, n in enumerate(sorted(insurers, key=lambda x: g.degree(x), reverse=True)):
            pos[n] = (0.0, -i * (10 / max(len(insurers), 1)))
        for i, n in enumerate(sorted(firms, key=lambda x: g.degree(x), reverse=True)):
            pos[n] = (6.0, -i * (10 / max(len(firms), 1)))
        for i, n in enumerate(others):
            pos[n] = (3.0, -i * (10 / max(len(others), 1)) - 1)
    else:
        pos = nx.spring_layout(g, k=0.6, seed=42, iterations=60)

    fig, ax = plt.subplots(figsize=figsize)

    strengths = {n: sum(d["weight"] for _, _, d in g.in_edges(n, data=True)) +
                    sum(d["weight"] for _, _, d in g.out_edges(n, data=True))
                 for n in g.nodes()}
    smax = max(strengths.values()) or 1.0
    sizes = [300 + 2200 * (math.log1p(strengths[n]) / math.log1p(smax)) for n in g.nodes()]
    colors = [PALETTE[g.nodes[n]["node_class"]] for n in g.nodes()]
    borders = ["#B00020" if g.nodes[n]["trust"] else "#FFFFFF" for n in g.nodes()]
    bwidths = [2.5 if g.nodes[n]["trust"] else 0.8 for n in g.nodes()]

    nx.draw_networkx_nodes(g, pos, node_size=sizes, node_color=colors,
                           edgecolors=borders, linewidths=bwidths, ax=ax)

    wmax = max((d["weight"] for _, _, d in g.edges(data=True)), default=1.0)
    for u, v, d in g.edges(data=True):
        nx.draw_networkx_edges(
            g, pos, edgelist=[(u, v)], ax=ax,
            edge_color=EDGE_COLORS[d["kind"]],
            width=0.6 + 4.0 * (math.log1p(d["weight"]) / math.log1p(wmax)),
            alpha=0.75, arrows=True, arrowsize=13,
            connectionstyle="arc3,rad=0.08",
        )

    labels = {n: (g.nodes[n]["name"][:28] + "…") if len(str(g.nodes[n]["name"])) > 28
              else g.nodes[n]["name"] for n in g.nodes()}
    nx.draw_networkx_labels(g, pos, labels, font_size=7.5, ax=ax)

    from matplotlib.lines import Line2D
    legend = [
        Line2D([0], [0], marker="o", color="w", label="Insurer",
               markerfacecolor=PALETTE["insurer"], markersize=11),
        Line2D([0], [0], marker="o", color="w", label="Law firm",
               markerfacecolor=PALETTE["law_firm"], markersize=11),
        Line2D([0], [0], marker="o", color="w", label="Trust account (red ring)",
               markerfacecolor=PALETTE["law_firm"], markeredgecolor="#B00020",
               markeredgewidth=2.5, markersize=11),
        Line2D([0], [0], color=EDGE_COLORS["firm_to_insurer"], lw=3,
               label="firm → insurer (RECOVERY)"),
        Line2D([0], [0], color=EDGE_COLORS["insurer_to_firm"], lw=3,
               label="insurer → firm (defense/funding)"),
        Line2D([0], [0], color=EDGE_COLORS["insurer_to_insurer"], lw=3,
               label="insurer ↔ insurer"),
    ]
    ax.legend(handles=legend, loc="upper left", fontsize=9, frameon=True)
    ax.set_title(f"{title}\n({g.number_of_nodes()} nodes, {g.number_of_edges()} edges) "
                 f"— node size ∝ log total amount, edge width ∝ log amount", fontsize=11)
    ax.axis("off")
    plt.tight_layout()
    return fig, ax

### 17b. Render the insurer <-> law-firm subgraph

The A/B-tier recovery candidates from section 12b plus the insurers they
touch. This is the picture to put in front of stakeholders: green edges
flowing right-to-left ARE the subrogation recovery story.

In [ ]:
cand_pairs = ins_law_pairs.loc[ins_law_pairs["confidence_tier"].isin(["A_high", "B_medium"])]
viz_ids = set(cand_pairs["insurer_id"]) | set(cand_pairs["firm_id"])
print(f"Law-firm recovery view: {len(viz_ids)} nodes "
      f"({cand_pairs['firm_id'].nunique()} firms, {cand_pairs['insurer_id'].nunique()} insurers)")

g_law = build_viz_graph(edges, nodes, viz_ids, top_n_edges=120, max_degree=None,
                        include_kinds={"firm_to_insurer", "insurer_to_firm"})
fig, ax = draw_static(g_law, title="Insurer ↔ law-firm subrogation recovery candidates",
                      bipartite=True)

### 17c. Render the carrier-to-carrier reciprocal core

The other half of the picture: the balanced carrier<->carrier corridors
from section 10. Spring layout here, since it isn't bipartite. Set
`max_degree` to pull out the clearinghouse hub and see the structure
underneath it.

In [ ]:
recip_ids = set(recip_tiered["source_fwd"]) | set(recip_tiered["dest_fwd"])
g_recip = build_viz_graph(edges, nodes, recip_ids, top_n_edges=120, max_degree=None)
fig2, ax2 = draw_static(g_recip, title="Carrier ↔ carrier reciprocal corridors",
                        bipartite=False)

### 17d. Combined view, hub suppressed

Insurers + law firms together, with the degree cap active. Compare against
17b/17c: a clearinghouse that touches every carrier dominates the layout
and hides the pairwise structure, which is exactly why the roadmap treats
hub handling as a prerequisite for any graph visual.

In [ ]:
combined_ids = viz_ids | recip_ids
g_all = build_viz_graph(edges, nodes, combined_ids, top_n_edges=200,
                        max_degree=HUB_DEGREE_CAP)
fig3, ax3 = draw_static(g_all, title=f"Subrogation subgraph (hubs > degree {HUB_DEGREE_CAP} suppressed)",
                        bipartite=False, figsize=(16, 11))

### 17e. Interactive view (pyvis)

Writes a standalone HTML file -- drag, zoom, hover for entity name, class,
NAICS, amount, months active. Better than the static view for actually
exploring; the static view is better for slides. Requires `pip install
pyvis`; skipped cleanly if unavailable.

In [ ]:
def draw_interactive(g, out_html="subrogation_network.html", height="800px"):
    try:
        from pyvis.network import Network
    except ImportError:
        print("pyvis not installed -- skipping interactive view. "
              "Install with: pip install pyvis --break-system-packages")
        return None

    if g.number_of_nodes() == 0:
        print("Empty graph -- nothing to render.")
        return None

    net = Network(height=height, width="100%", directed=True,
                  bgcolor="#FFFFFF", font_color="#222222", notebook=False,
                  # 'in_line' embeds vis-network JS directly in the HTML, so the
                  # output is ONE self-contained file that opens offline. The
                  # default ('local') writes a sibling lib/ folder the HTML then
                  # depends on, and 'remote' needs a CDN that's usually blocked
                  # on an internal network -- neither travels well by email.
                  cdn_resources="in_line")
    net.barnes_hut(gravity=-9000, central_gravity=0.25, spring_length=180)

    strengths = {n: sum(d["weight"] for _, _, d in g.in_edges(n, data=True)) +
                    sum(d["weight"] for _, _, d in g.out_edges(n, data=True))
                 for n in g.nodes()}
    smax = max(strengths.values()) or 1.0

    for n, d in g.nodes(data=True):
        title = (f"{d['name']}\nclass: {d['node_class']}"
                 f"\ntrust account: {d['trust']}"
                 f"\ntotal amount: {strengths[n]:,.0f}")
        net.add_node(
            n, label=str(d["name"])[:30], title=title,
            color={"background": PALETTE[d["node_class"]],
                   "border": "#B00020" if d["trust"] else "#666666"},
            borderWidth=3 if d["trust"] else 1,
            size=12 + 30 * (math.log1p(strengths[n]) / math.log1p(smax)),
        )

    wmax = max((d["weight"] for _, _, d in g.edges(data=True)), default=1.0)
    for u, v, d in g.edges(data=True):
        net.add_edge(
            u, v, color=EDGE_COLORS[d["kind"]],
            width=1 + 7 * (math.log1p(d["weight"]) / math.log1p(wmax)),
            title=(f"{d['kind']}\namount: {d['weight']:,.0f}"
                   f"\nvolume: {d['volume']:,.0f}\nmonths active: {d['months']}"),
        )

    out_path = Path(out_html)
    net.write_html(str(out_path), notebook=False)
    print(f"Wrote interactive graph -> {out_path.resolve()}")
    return out_path


html_path = draw_interactive(g_all, out_html=str(OUT_DIR / "subrogation_network.html"))

# In Jupyter, display it inline:
# from IPython.display import IFrame
# IFrame(str(html_path), width="100%", height="820px")